# Proyecto MiBici: propuesta y línea base reproducible

Este notebook inicia la entrega del proyecto integrador. Documenta una propuesta para las tres preguntas de negocio y deja preparada una base reproducible para ejecutar los experimentos desde código.

**Preguntas:**
- **P1 — Regresión:** ¿cuántos viajes se iniciarán por estación y día?
- **P2 — Clasificación:** ¿una estación estará en riesgo de saturación o de quedarse sin bicicletas/anclajes?
- **P3 — Clustering:** ¿qué grupos de estaciones comparten patrones de uso y dónde se ubican?

Las celdas que requieren CSV locales no fallan si aún no se han descargado: muestran el siguiente paso y permiten continuar con la documentación.

## 1. Decisiones iniciales de modelado

| Pregunta | Unidad y horizonte | Variable objetivo / salida | Baseline inicial |
|---|---|---|---|
| P1 | estación-día; siguiente día | número de viajes iniciados | media histórica de la estación |
| P2 | estación-día; siguiente día | riesgo alto (1/0), aproximado con demanda y capacidad | clase mayoritaria |
| P3 | estación; periodo completo | grupo de uso y ubicación | K-Means sobre variables estandarizadas |

La partición de P1 y P2 será temporal: entrenamiento en fechas anteriores, validación en un bloque posterior y prueba en el bloque más reciente. No se mezclan observaciones futuras con el entrenamiento.

In [ ]:
# Configuración reproducible
from pathlib import Path
import re
import unicodedata

SEED = 42
TEAM_ID = "equipo23"
START_DATE = "2019-01-01"
END_DATE = "2026-09-30"

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "reports" / "figures"
for folder in (DATA_RAW, DATA_PROCESSED, FIGURES):
    folder.mkdir(parents=True, exist_ok=True)

print(f"Proyecto: {PROJECT_ROOT}")
print(f"Rango: {START_DATE} a {END_DATE} | semilla: {SEED}")


In [ ]:
# Dependencias mínimas. Si falta alguna, instalarla en el entorno del proyecto.
from importlib.util import find_spec

required = ["pandas", "numpy", "matplotlib", "sklearn"]
missing = [package for package in required if find_spec(package) is None]
if missing:
    print("Faltan dependencias:", ", ".join(missing))
    print("Instalar con: pip install pandas numpy matplotlib scikit-learn")
else:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.cluster import KMeans
    from sklearn.dummy import DummyClassifier, DummyRegressor
    from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, f1_score, silhouette_score
    from sklearn.preprocessing import StandardScaler
    print("Dependencias disponibles.")


## 2. Ingesta y normalización reproducible

Los archivos de viajes se guardan en `data/raw/` y el catálogo de estaciones en la misma carpeta. La lista de archivos debe quedar versionada en el futuro módulo de ingesta; los CSV grandes no se deben subir al repositorio. La detección de encoding permite manejar las variaciones históricas del portal.

In [ ]:
def normalize_column(name: str) -> str:
    """Convierte nombres históricos a una forma comparable."""
    text = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode()
    text = re.sub(r"[^a-zA-Z0-9]+", "_", text).strip("_").lower()
    return text


def read_csv_robust(path: Path) -> pd.DataFrame:
    """Lee un CSV probando encodings comunes y normaliza sus columnas."""
    last_error = None
    for encoding in ("utf-8", "utf-8-sig", "cp1252", "latin1"):
        try:
            frame = pd.read_csv(path, encoding=encoding, low_memory=False)
            frame.columns = [normalize_column(column) for column in frame.columns]
            return frame
        except UnicodeDecodeError as error:
            last_error = error
    raise UnicodeDecodeError("unknown", b"", 0, 1, f"No se pudo leer {path}: {last_error}")


travel_files = sorted(DATA_RAW.glob("datos_abiertos_*.csv"))
station_files = sorted(DATA_RAW.glob("nomenclatura_*.csv"))
if not travel_files or not station_files:
    print("Aún no hay datos locales.")
    print("Coloca los CSV de viajes como data/raw/datos_abiertos_AAAA_MM.csv")
    print("y el catálogo como data/raw/nomenclatura_AAAA_MM.csv; después ejecuta de nuevo esta celda.")
else:
    trips = pd.concat((read_csv_robust(path) for path in travel_files), ignore_index=True)
    stations = read_csv_robust(station_files[-1])
    print(f"Viajes: {len(trips):,} | estaciones: {len(stations):,}")
    print("Columnas de viajes:", trips.columns.tolist())
    print("Columnas de estaciones:", stations.columns.tolist())


In [ ]:
# Exploración inicial, solo si los datos fueron cargados.
    if "trips" in globals():
        display(trips.head())
        display(trips.isna().mean().sort_values(ascending=False).head(15).to_frame("fraccion_nula"))
        print(trips.describe(include="all").T.head(20))


## 3. Feature engineering sin fuga temporal

Se requiere adaptar los nombres de fecha y estación a los nombres que entregue cada versión del portal. Las variables históricas (`lag_1`, `lag_7`, medias móviles) se calculan únicamente con valores previos; el objetivo se desplaza un día hacia adelante. Para P2, el riesgo es una aproximación operacional: se marca alto cuando la demanda esperada supera una fracción de la capacidad. Debe validarse y justificarse con el equipo porque no existe inventario real de bicicletas en los datos.

In [ ]:
def build_daily_features(trips: pd.DataFrame, stations: pd.DataFrame) -> pd.DataFrame:
    """Construye una tabla estación-día para P1 y P2."""
    date_candidates = ["fecha_inicio", "fecha_hora_inicio", "start_time", "inicio"]
    station_candidates = ["id_estacion_inicio", "estacion_inicio", "start_station_id", "origen"]
    date_col = next((column for column in date_candidates if column in trips), None)
    station_col = next((column for column in station_candidates if column in trips), None)
    if date_col is None or station_col is None:
        raise KeyError(f"No se encontraron fecha/estación de inicio. Columnas: {trips.columns.tolist()}")
    result = trips[[date_col, station_col]].copy()
    result["fecha"] = pd.to_datetime(result[date_col], errors="coerce").dt.floor("D")
    result["estacion_id"] = result[station_col].astype("string")
    result = result.dropna(subset=["fecha", "estacion_id"])
    daily = result.groupby(["fecha", "estacion_id"], as_index=False).size()
    daily = daily.rename(columns={"size": "viajes_iniciados"}).sort_values(["estacion_id", "fecha"])
    grouped = daily.groupby("estacion_id")["viajes_iniciados"]
    daily["viajes_lag_1"] = grouped.shift(1)
    daily["viajes_lag_7"] = grouped.shift(7)
    daily["media_7_dias"] = grouped.transform(lambda values: values.shift(1).rolling(7, min_periods=3).mean())
    daily["dia_semana"] = daily["fecha"].dt.dayofweek
    daily["mes"] = daily["fecha"].dt.month
    # Capacidad: buscar una columna razonable; si no existe, dejar explícita la limitación.
    capacity_col = next((c for c in ("capacidad", "capacity", "capacidad_estacion") if c in stations), None)
    station_id_col = next((c for c in ("id_estacion", "estacion_id", "station_id", "id") if c in stations), None)
    if capacity_col and station_id_col:
        capacity = stations[[station_id_col, capacity_col]].rename(columns={station_id_col: "estacion_id", capacity_col: "capacidad"})
        capacity["estacion_id"] = capacity["estacion_id"].astype("string")
        daily = daily.merge(capacity, on="estacion_id", how="left")
        daily["riesgo_alto"] = (daily["viajes_lag_1"] / daily["capacidad"] >= 0.8).astype("Int64")
        daily["riesgo_alto_siguiente"] = (daily["objetivo_viajes_siguiente"] / daily["capacidad"] >= 0.8).astype("Int64")
    else:
        daily["capacidad"] = pd.NA
        daily["riesgo_alto"] = pd.NA
        daily["riesgo_alto_siguiente"] = pd.NA
    return daily


if "trips" in globals() and "stations" in globals():
    daily_features = build_daily_features(trips, stations)
    display(daily_features.head())
else:
    print("Carga los CSV para construir las variables de P1 y P2.")


In [ ]:
# Baselines temporales de P1 y P2.
    if "daily_features" in globals():
        model_data = daily_features.dropna(subset=["viajes_lag_1", "viajes_lag_7"]).copy()
        cutoff = model_data["fecha"].quantile(0.8)
        train = model_data[model_data["fecha"] < cutoff]
        test = model_data[model_data["fecha"] >= cutoff]
        print(f"Entrenamiento: {len(train):,} | prueba temporal: {len(test):,} | corte: {cutoff.date()}")
        p1_baseline = train.groupby("estacion_id")["viajes_iniciados"].mean()
        p1_pred = test["estacion_id"].map(p1_baseline).fillna(train["viajes_iniciados"].mean())
        print("P1 MAE baseline:", mean_absolute_error(test["viajes_iniciados"], p1_pred))
        if test["riesgo_alto_siguiente"].notna().any():
            p2_target = train["riesgo_alto_siguiente"].dropna()
            majority = int(p2_target.mode().iloc[0])
            p2_pred = np.full(test["riesgo_alto_siguiente"].notna().sum(), majority)
            p2_test = test["riesgo_alto_siguiente"].dropna()
            print("P2 accuracy baseline:", accuracy_score(p2_test, p2_pred))
            print("P2 F1 baseline:", f1_score(p2_test, p2_pred, zero_division=0))
    else:
        print("Baselines pendientes: no hay datos locales.")


In [ ]:
# P3: variables por estación y clustering exploratorio.
    if "daily_features" in globals():
        station_features = daily_features.groupby("estacion_id").agg(
            viajes_promedio=("viajes_iniciados", "mean"),
            viajes_desviacion=("viajes_iniciados", "std"),
            dias_observados=("fecha", "nunique"),
            capacidad=("capacidad", "first"),
        ).fillna(0)
        X = StandardScaler().fit_transform(station_features)
        k = min(4, len(station_features))
        if k >= 2:
            clusters = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit_predict(X)
            station_features["cluster"] = clusters
            print("Silhouette P3:", silhouette_score(X, clusters))
            display(station_features.groupby("cluster").mean(numeric_only=True))
    else:
        print("Clustering pendiente: no hay datos locales.")


## 4. Registro de experimentos en MLflow

Los runs definitivos deben ejecutarse desde el módulo del proyecto, no solo desde este notebook. Esta celda sirve como contrato de parámetros, métricas, etiquetas y artefactos. El commit de Git debe capturarse al ejecutar el script de entrenamiento.

In [ ]:
if find_spec("mlflow") is None:
    print("MLflow no está instalado. Instalar con: pip install mlflow")
else:
    import mlflow
    mlflow.set_experiment(f"mibici-{TEAM_ID}")
    with mlflow.start_run(run_name="p1-regresion-baseline"):
        mlflow.log_params({"seed": SEED, "start_date": START_DATE, "end_date": END_DATE, "data_version": "references/data_manifest.csv"})
        mlflow.set_tags({"team": TEAM_ID, "question": "P1", "git_commit": "pendiente"})
        print("Run P1 preparado; conectar aquí las métricas y figuras del entrenamiento.")


## 5. Próximos pasos para completar la entrega

1. Crear el repositorio con Cookiecutter Data Science y mover este notebook a `notebooks/`.
2. Descargar los CSV seleccionados y verificar su versión contra `references/data_manifest.csv`; documentar meses, viajes y estaciones reales.
3. Confirmar los nombres de columnas de cada periodo y completar el módulo de ingesta.
4. Implementar `config.py`, `features.py` y scripts de entrenamiento para P1, P2 y P3.
5. Registrar un run por pregunta en MLflow, incluyendo parámetros, métricas, tags, commit y artefactos.
6. Añadir en `references/` el ML Canvas, el diagrama de arquitectura y las condiciones de uso/datos personales.
7. Documentar en el README el flujo de ramas, Pull Requests, revisión y reproducibilidad.

**Limitación clave:** P2 no observa inventario real. La etiqueta de riesgo es un proxy basado en demanda y capacidad; debe presentarse como supuesto, validarse con el equipo y revisarse si se obtiene información adicional.